# SepsisGuard — GRPO Training Notebook

Train 4 multi-agent roles (Nurse, Lab, Pharmacist, Physician) using TRL GRPO with Unsloth 4-bit quantization.

This notebook:
1. Loads a quantized Qwen 2.5-3B model
2. Connects to the live SepsisGuard environment
3. Collects initial rollouts for the prompt dataset
4. Runs GRPO training with online environment rewards
5. Evaluates and plots reward improvement vs heuristic baseline

In [1]:
!pip install -q -U "unsloth[colab-new]" openenv-core "trl>=0.12" vllm datasets matplotlib
!pip install -q requests httpx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=4096, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)
FastLanguageModel.for_inference(model)
print(f"Model loaded: {MODEL_NAME}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded: unsloth/Qwen2.5-3B-Instruct-bnb-4bit


In [3]:
import os
if not os.path.exists("/content/sepsisguard"):
    !git clone https://huggingface.co/spaces/Jishnu-Vijayan-03/Sepsis-Guard /content/sepsisguard
os.chdir("/content/sepsisguard")

import sys
if "/content/sepsisguard" not in sys.path:
    sys.path.insert(0, "/content/sepsisguard")

!pip install -q -e "/content/sepsisguard"
print("Repo ready:", os.listdir("/content/sepsisguard"))

Cloning into '/content/sepsisguard'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 95 (delta 42), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (95/95), 259.45 KiB | 2.47 MiB/s, done.
Resolving deltas: 100% (42/42), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sepsisguard (pyproject.toml) ... done
Repo ready: ['pyproject.toml', 'server', 'training', 'requirements.txt', 'Dockerfile', 'inference.py', 'README.md', '.env.example', 'models.py', 'tests', '.gitignore', 'agents', '.gitattributes', 'openenv.yaml', 'uv.lock', 'CLAUDE.md', '.git']


In [5]:
!pip install -q -e "/content/sepsisguard"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sepsisguard (pyproject.toml) ... done


In [21]:
%cd /content/sepsisguard
import os
import signal
import subprocess
import time
import requests

UVICORN_CMD = "uvicorn server.app:app --host 0.0.0.0 --port 7860"
HEALTH_URL = "http://127.0.0.1:7860/health"

def _find_server_pids():
    # Match the specific SepsisGuard uvicorn command to avoid killing unrelated processes.
    proc = subprocess.run(
        ["bash", "-lc", f"pgrep -f '{UVICORN_CMD}'"],
        capture_output=True, text=True
    )
    if proc.returncode != 0 or not proc.stdout.strip():
        return []
    return [int(x) for x in proc.stdout.strip().splitlines() if x.strip().isdigit()]

def _stop_existing_server():
    pids = _find_server_pids()
    if not pids:
        print("No existing SepsisGuard uvicorn process found.")
        return
    print(f"Stopping existing uvicorn process(es): {pids}")
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except OSError:
            pass
    time.sleep(1.5)

    # Force kill if still alive.
    survivors = _find_server_pids()
    for pid in survivors:
        try:
            os.kill(pid, signal.SIGKILL)
        except OSError:
            pass
    if survivors:
        print(f"Force-killed lingering process(es): {survivors}")

def _start_server():
    subprocess.run(
        ["bash", "-lc", f"nohup {UVICORN_CMD} > /content/uvicorn.log 2>&1 &"],
        check=False
    )

def _wait_for_health(timeout_s=30):
    t0 = time.time()
    last_err = None
    while time.time() - t0 < timeout_s:
        try:
            r = requests.get(HEALTH_URL, timeout=3)
            if r.ok:
                return True
        except Exception as e:
            last_err = e
        time.sleep(1)
    print(f"Server health check timed out at {HEALTH_URL}. Last error: {last_err}")
    return False

_stop_existing_server()
_start_server()
if _wait_for_health(timeout_s=30):
    print("Local SepsisGuard server is healthy at http://127.0.0.1:7860")
else:
    print("Server started but health check failed. Inspect logs: /content/uvicorn.log")

/content
No existing SepsisGuard uvicorn process found.
Local SepsisGuard server is healthy at http://127.0.0.1:7860


In [22]:
!curl "http://127.0.0.1:7860/health"

{"status":"healthy"}

In [23]:
import os, requests, time

# If notebook and server run on the SAME Colab VM, localhost works.
# If your server is outside Colab, set ENV_BASE_URL to a public tunnel URL.
ENV_URL = os.environ.get("ENV_BASE_URL", "http://127.0.0.1:7860")

class EnvClient:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip("/")

    def reset(self, task_name, seed, session_id=None):
        headers = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/reset",
                          json={"task_name": task_name, "seed": seed},
                          headers=headers, timeout=30)
        r.raise_for_status()
        return r.json()

    def step(self, actions, session_id=None):
        headers = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/step",
                          json={"actions": actions},
                          headers=headers, timeout=30)
        r.raise_for_status()
        return r.json()

    def create_session(self):
        r = requests.post(f"{self.base_url}/session", timeout=10)
        r.raise_for_status()
        return r.json()["session_id"]

    def delete_session(self, session_id):
        try:
            requests.delete(f"{self.base_url}/session/{session_id}", timeout=5)
        except Exception:
            pass

# Wait briefly for server readiness so first /reset doesn't fail noisily.
ready = False
for _ in range(15):
    try:
        requests.get(f"{ENV_URL}/state", timeout=3).raise_for_status()
        ready = True
        break
    except Exception:
        time.sleep(1)
if not ready:
    raise RuntimeError(
        f"Server not reachable at {ENV_URL}. Start uvicorn or set ENV_BASE_URL to your reachable URL."
    )

env = EnvClient(ENV_URL)
info = env.reset(task_name="task1_textbook", seed=42)
print(f"Connected to {ENV_URL}")
print(f"Tick: {info['info']['tick']}, Roles: {list(info['observations'].keys())}")

Connected to http://127.0.0.1:7860
Tick: 1, Roles: ['nurse', 'lab', 'pharmacist', 'physician']


In [ ]:
import os, sys, json
from tqdm.auto import tqdm
sys.path.insert(0, "/content/sepsisguard")

from training.prompts import build_role_prompt
from agents.nurse import HeuristicNurse
from agents.lab import HeuristicLab
from agents.pharmacist import HeuristicPharmacist
from agents.physician import HeuristicPhysician

ROLES = ("nurse", "lab", "pharmacist", "physician")
N_EPISODES = 16
SEEDS = list(range(42, 42 + N_EPISODES))
TASK = "task1_textbook"
MAX_TICKS_PER_EP = 48

heuristic_agents = {
    "nurse": HeuristicNurse(),
    "lab": HeuristicLab(),
    "pharmacist": HeuristicPharmacist(),
    "physician": HeuristicPhysician(),
}

# GRPO only uses prompts — TRL generates its own completions from the evolving policy.
# We collect diverse prompts by stepping through episodes with heuristic agents.
# No LLM inference needed, so 16 episodes takes ~30 seconds instead of 60+ minutes.
print(f"Collecting prompts via {N_EPISODES} heuristic episodes...")
rollouts = []

for ep_idx, seed in enumerate(tqdm(SEEDS, desc="Episodes")):
    sid = env.create_session()
    bundle = env.reset(task_name=TASK, seed=seed, session_id=sid)
    done = False
    tick = 0

    while not done and tick < MAX_TICKS_PER_EP:
        obs = bundle["observations"]
        actions = {}
        for role in ROLES:
            prompt = build_role_prompt(obs[role], role)
            actions[role] = heuristic_agents[role].decide(obs[role])
            rollouts.append({"prompt": prompt, "role": role, "seed": seed, "tick": tick + 1})
        bundle = env.step(actions, session_id=sid)
        done = bool(bundle.get("done", False))
        tick += 1

    env.delete_session(sid)

print(f"\nCollected {len(rollouts)} prompts from {N_EPISODES} episodes")
print(f"Roles: {set(r['role'] for r in rollouts)}")
print(f"Avg ticks/episode: {len(rollouts) / N_EPISODES / len(ROLES):.0f}")

In [10]:
!curl "http://127.0.0.1:7860/health"

{"status":"healthy"}

In [ ]:
import json
from datasets import Dataset

def get_clinical_state(prompt_str):
    """Strips out tick counters and rewards to find the true clinical state for dedup."""
    try:
        obs_start = prompt_str.find('Observation:\n') + 13
        obs_end = prompt_str.rfind('\nAction (JSON):')
        if obs_start < 13 or obs_end == -1: return prompt_str
        obs_json_str = prompt_str[obs_start:obs_end]
        obs_dict = json.loads(obs_json_str)
        for key in ["tick", "reward", "cumulative_reward", "last_action_result",
                     "normalized_score", "done", "metadata"]:
            obs_dict.pop(key, None)
        clean_obs_str = json.dumps(obs_dict, sort_keys=True)
        return prompt_str[:obs_start] + clean_obs_str + prompt_str[obs_end:]
    except Exception:
        return prompt_str

unique_prompts = {}
for r in rollouts:
    core_state = get_clinical_state(r["prompt"])
    if core_state not in unique_prompts:
        unique_prompts[core_state] = r["prompt"]

train_dataset = Dataset.from_list([{"prompt": p} for p in unique_prompts.values()])

print(f"Original rollouts: {len(rollouts)}")
print(f"Unique clinical scenarios: {len(unique_prompts)}")

# --- PRE-TRAINING EVALUATION (before GRPO changes the model) ---
# This establishes the "before" baseline so we can show improvement.
import requests, re, torch
from training.prompts import build_role_prompt

nurse_h, lab_h, pharma_h, phys_h = HeuristicNurse(), HeuristicLab(), HeuristicPharmacist(), HeuristicPhysician()
heuristic_agents_eval = {"nurse": nurse_h, "lab": lab_h, "pharmacist": pharma_h, "physician": phys_h}

def run_episode(env_client, task_name, seed, agent_fn, session_id=None):
    bundle = env_client.reset(task_name=task_name, seed=seed, session_id=session_id)
    done = False
    while not done:
        obs = bundle["observations"]
        actions = {role: agent_fn(role, obs[role]) for role in ("nurse", "lab", "pharmacist", "physician")}
        bundle = env_client.step(actions, session_id=session_id)
        done = bundle["done"]
    grader = requests.get(f"{env_client.base_url}/grader",
                          headers={"X-Session-Id": session_id} if session_id else {},
                          timeout=30).json()
    return grader.get("score", 0.0)

def heuristic_agent_fn(role, obs):
    return heuristic_agents_eval[role].decide(obs)

def make_llm_agent_fn(model, tokenizer, target_role):
    def agent_fn(role, obs):
        if role != target_role:
            return heuristic_agents_eval[role].decide(obs)
        prompt = build_role_prompt(obs, role)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        try:
            parsed = json.loads(text.strip())
            if isinstance(parsed, dict) and "operation" in parsed: return parsed
        except Exception: pass
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
                if isinstance(parsed, dict) and "operation" in parsed: return parsed
            except Exception: pass
        return heuristic_agents_eval[role].decide(obs)
    return agent_fn

N_EVAL = 5
EVAL_SEEDS = list(range(100, 100 + N_EVAL))

FastLanguageModel.for_inference(model)

print("\n" + "=" * 60)
print("PRE-TRAINING EVALUATION (before GRPO)")
print("=" * 60)

pre_baseline_scores = []
for seed in EVAL_SEEDS:
    sid = env.create_session()
    score = run_episode(env, "task1_textbook", seed, heuristic_agent_fn, session_id=sid)
    pre_baseline_scores.append(score)
    env.delete_session(sid)
print(f"Heuristic baseline: mean={sum(pre_baseline_scores)/len(pre_baseline_scores):.4f}  scores={[round(s,3) for s in pre_baseline_scores]}")

pre_trained_scores = {}
for target_role in ("nurse", "lab", "pharmacist", "physician"):
    llm_fn = make_llm_agent_fn(model, tokenizer, target_role)
    scores = []
    for seed in EVAL_SEEDS:
        sid = env.create_session()
        score = run_episode(env, "task1_textbook", seed, llm_fn, session_id=sid)
        scores.append(score)
        env.delete_session(sid)
    pre_trained_scores[target_role] = scores
    print(f"Pre-train [{target_role}]: mean={sum(scores)/len(scores):.4f}  scores={[round(s,3) for s in scores]}")

print("=" * 60)

In [ ]:
import torch
import json
import re
import time
import requests
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback
from training.reward_shaping import make_online_sepsis_reward_fn, format_reward_fn

FastLanguageModel.for_training(model)

# --- 1. HARDWARE SETUP ---
has_cuda = torch.cuda.is_available()
if has_cuda:
    major, _ = torch.cuda.get_device_capability()
    use_bf16 = major >= 8
    use_fp16 = not use_bf16
    print(f"CUDA device: {torch.cuda.get_device_name(0)} | bf16={use_bf16} fp16={use_fp16}")
else:
    use_bf16 = False
    use_fp16 = False
    print("CUDA not available. Using fp32 precision.")


# --- 2. GRPO CONFIGURATION ---
cfg = GRPOConfig(
    output_dir="./sepsis-grpo",
    num_generations=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_steps=120,
    learning_rate=5e-6,
    warmup_steps=12,
    logging_steps=1,
    save_steps=40,
    max_prompt_length=3000,
    max_completion_length=128,
    temperature=0.6,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    gradient_checkpointing=True,
)


# --- 3. REWARD FUNCTIONS ---
# Use a denser online reward window with low concurrency for stability.
reward_fn_env_obj = make_online_sepsis_reward_fn(
    env_url=ENV_URL,
    task_name=TASK,
    seed=42,
    warmup_ticks=6,
    inject_ticks=6,
    max_workers=1,
)

print("=" * 50)
print("PRE-FLIGHT REWARD CHECK")
print("=" * 50)
test_prompts = [train_dataset[0]["prompt"], train_dataset[0]["prompt"]]
test_completions = [
    '{"operation": "noop"}',
    '{"operation": "escalate_to_physician", "patient_id": "P01", "urgency": "critical", "rationale": "HR 130 BP 80 temp 39.1"}'
]
test_env_rewards = reward_fn_env_obj(test_completions, test_prompts)
test_fmt_rewards = format_reward_fn(test_completions, test_prompts)
print(f"Env rewards:    noop={test_env_rewards[0]:.4f}  escalate={test_env_rewards[1]:.4f}")
print(f"Format rewards: noop={test_fmt_rewards[0]:.4f}  escalate={test_fmt_rewards[1]:.4f}")
if test_env_rewards[0] == 0.0 and test_env_rewards[1] == 0.0:
    print("🔴 BOTH ARE 0.0 — DO NOT TRAIN")
else:
    print("✅ Reward function is working")
print("=" * 50)

def reward_fn_env(*args, **kwargs):
    return reward_fn_env_obj(*args, **kwargs)
reward_fn_env.__name__ = "reward_fn_env"

reward_log = {"steps": [], "env_reward": [], "format_reward": [], "combined_reward": []}


# --- 4. REWARD LOGGER WITH COLLAPSE DETECTION + SERVER HEALTH ---
class RewardLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step

        # Capture rewards from any available TRL logging keys
        env_r = (logs.get("rewards/reward_fn_env/mean")
                 or logs.get("reward_fn_env", 0.0))
        fmt_r = (logs.get("rewards/format_reward_fn/mean")
                 or logs.get("format_reward_fn", 0.0))
        combined = logs.get("reward", env_r + fmt_r)

        if env_r != 0.0 or fmt_r != 0.0 or "reward" in logs:
            reward_log["steps"].append(step)
            reward_log["env_reward"].append(float(env_r))
            reward_log["format_reward"].append(float(fmt_r))
            reward_log["combined_reward"].append(float(combined))

        # Sample generation + server health every 50 steps
        if step % 50 == 0 and step > 0:
            # Server health check
            try:
                r = requests.get(f"{ENV_URL}/health", timeout=5)
                if not r.ok:
                    print(f"\n[WARN step {step}] Server unhealthy: {r.status_code}")
            except Exception as e:
                print(f"\n[WARN step {step}] Server unreachable: {e}")

            # Sample model output
            sample_prompt = train_dataset[0]["prompt"]
            inputs = tokenizer(sample_prompt, return_tensors="pt").to(model.device)
            model.eval()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            model.train()

            generated = tokenizer.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )

            clean_json = "Parse Failed - No JSON found"
            match = re.search(r'\{.*?\}', generated, re.DOTALL)
            if match:
                clean_json = match.group(0)

            collapse_warning = ""
            if '"..."' in clean_json or '"rationale": ""' in clean_json:
                collapse_warning = " [WARNING: possible mode collapse]"
            if clean_json.count('"noop"') > 0 or clean_json.count('"do_nothing"') > 0:
                collapse_warning += " [WARNING: noop action]"

            print(f"\n=== Step {step} sample{collapse_warning} ===")
            print(f"Extracted: {clean_json[:300]}")
            if reward_log["steps"]:
                last_env = reward_log["env_reward"][-1]
                last_fmt = reward_log["format_reward"][-1]
                print(f"Latest rewards: env={last_env:.3f} fmt={last_fmt:.3f}")
            print("=" * 40)


trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn_env, format_reward_fn],
    args=cfg,
    train_dataset=train_dataset,
    callbacks=[RewardLogger()],
)

print(f"Training: {cfg.max_steps} steps, lr={cfg.learning_rate}, "
      f"batch={cfg.per_device_train_batch_size}x{cfg.gradient_accumulation_steps}, "
      f"gen={cfg.num_generations}, grad_ckpt={cfg.gradient_checkpointing}")
# print(f"Reward: online env (inject_ticks=4,  # 4 ticks for lower variance than 1, still fast with prompt-hash seeding prompt-hash seed) + format")
print(f"Dataset: {len(train_dataset)} unique prompts")
print("Starting GRPO training...")
trainer.train()
print("Training complete.")

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
import os
import shutil

# --- 1. Define Paths ---
# Local Colab Paths
local_lora_path = "./sepsis-grpo-lora"
local_merged_path = "./sepsis-grpo-merged"

# Google Drive Paths
drive_base_dir = "/content/drive/MyDrive/sepsis_model_exports"
drive_lora_path = os.path.join(drive_base_dir, "sepsis-grpo-lora")
drive_merged_path = os.path.join(drive_base_dir, "sepsis-grpo-merged")

os.makedirs(drive_base_dir, exist_ok=True)

# --- 2. Save Locally (Fastest) ---
print(f"💾 Saving LoRA adapters locally to {local_lora_path} ...")
model.save_pretrained(local_lora_path)
tokenizer.save_pretrained(local_lora_path)

print(f"💾 Saving Merged 16-bit model locally to {local_merged_path} ...")
print("(This may take a few minutes...)")
model.save_pretrained_merged(
    local_merged_path,
    tokenizer,
    save_method="merged_16bit",
)
print("✅ Local save complete!")

# --- 3. Copy to Google Drive (Safe Backup) ---
print(f"\n📂 Copying LoRA adapters to Drive: {drive_lora_path} ...")
if os.path.exists(drive_lora_path):
    shutil.rmtree(drive_lora_path) # Overwrite if it already exists
shutil.copytree(local_lora_path, drive_lora_path)

print(f"📂 Copying Merged model to Drive: {drive_merged_path} ...")
print("(Copying 6GB+ to Drive... please wait...)")
if os.path.exists(drive_merged_path):
    shutil.rmtree(drive_merged_path)
shutil.copytree(local_merged_path, drive_merged_path)

print("\n🎉 Success! Models are saved locally and fully backed up to Google Drive.")

💾 Saving LoRA adapters locally to ./sepsis-grpo-lora ...
💾 Saving Merged 16-bit model locally to ./sepsis-grpo-merged ...
(This may take a few minutes...)


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [03:06<00:00, 93.41s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:25<00:00, 102.61s/it]


Unsloth: Merge process complete. Saved to `/content/sepsisguard/sepsis-grpo-merged`
✅ Local save complete!

📂 Copying LoRA adapters to Drive: /content/drive/MyDrive/sepsis_model_exports/sepsis-grpo-lora ...
📂 Copying Merged model to Drive: /content/drive/MyDrive/sepsis_model_exports/sepsis-grpo-merged ...
(Copying 6GB+ to Drive... please wait...)

🎉 Success! Models are saved locally and fully backed up to Google Drive.


In [ ]:
import requests, json, re, torch
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)

print("=" * 60)
print("POST-TRAINING EVALUATION")
print("=" * 60)

# --- Per-role evaluation (LLM replaces one role, heuristics for rest) ---
post_trained_scores = {}
for target_role in ("nurse", "lab", "pharmacist", "physician"):
    llm_fn = make_llm_agent_fn(model, tokenizer, target_role)
    scores = []
    for seed in EVAL_SEEDS:
        sid = env.create_session()
        score = run_episode(env, "task1_textbook", seed, llm_fn, session_id=sid)
        scores.append(score)
        env.delete_session(sid)
    post_trained_scores[target_role] = scores
    pre_mean = sum(pre_trained_scores[target_role]) / len(pre_trained_scores[target_role])
    post_mean = sum(scores) / len(scores)
    delta = post_mean - pre_mean
    print(f"Post-train [{target_role}]: mean={post_mean:.4f}  delta_vs_pre={delta:+.4f}  scores={[round(s,3) for s in scores]}")

# --- All-LLM evaluation (all 4 roles are LLM simultaneously) ---
def all_llm_agent_fn(role, obs):
    return make_llm_agent_fn(model, tokenizer, role)(role, obs)

all_llm_scores = []
for seed in EVAL_SEEDS:
    sid = env.create_session()
    score = run_episode(env, "task1_textbook", seed, all_llm_agent_fn, session_id=sid)
    all_llm_scores.append(score)
    env.delete_session(sid)
print(f"\nAll-LLM (4 roles): mean={sum(all_llm_scores)/len(all_llm_scores):.4f}  scores={[round(s,3) for s in all_llm_scores]}")

# --- Summary table ---
heuristic_mean = sum(pre_baseline_scores) / len(pre_baseline_scores)
print(f"\n{'Role':<15} {'Heuristic':>10} {'Pre-Train':>10} {'Post-Train':>10} {'Delta':>10}")
print("-" * 60)
for role in ("nurse", "lab", "pharmacist", "physician"):
    pre = sum(pre_trained_scores[role]) / len(pre_trained_scores[role])
    post = sum(post_trained_scores[role]) / len(post_trained_scores[role])
    delta = post - pre
    print(f"{role:<15} {heuristic_mean:>10.4f} {pre:>10.4f} {post:>10.4f} {delta:>+10.4f}")
all_post = sum(all_llm_scores) / len(all_llm_scores)
print(f"{'all-llm':<15} {heuristic_mean:>10.4f} {'n/a':>10} {all_post:>10.4f} {'':>10}")
print("=" * 60)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# --- Panel 1: Reward Curves ---
if reward_log["steps"]:
    axes[0].plot(reward_log["steps"], reward_log["env_reward"], label="Env Reward", linewidth=1.5, alpha=0.8)
    axes[0].plot(reward_log["steps"], reward_log["format_reward"], label="Format Reward", linewidth=1.5, alpha=0.8)
    if reward_log["combined_reward"]:
        axes[0].plot(reward_log["steps"], reward_log["combined_reward"], label="Combined", linewidth=2, color="black", alpha=0.5)
    axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.3)
    axes[0].set_xlabel("Training Step")
    axes[0].set_ylabel("Mean Reward")
    axes[0].set_title("GRPO Training Reward Curves")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "No reward logs captured\n(training may not have run yet)",
                 ha="center", va="center", transform=axes[0].transAxes, fontsize=11)
    axes[0].set_title("GRPO Training Reward Curves")

# --- Panel 2: Before vs After (per role) ---
roles = ["nurse", "lab", "pharmacist", "physician"]
heuristic_mean = sum(pre_baseline_scores) / len(pre_baseline_scores)

pre_means = [sum(pre_trained_scores[r]) / len(pre_trained_scores[r]) for r in roles]
post_means = [sum(post_trained_scores[r]) / len(post_trained_scores[r]) for r in roles]

x = np.arange(len(roles))
width = 0.35
bars_pre = axes[1].bar(x - width/2, pre_means, width, label="Pre-Training", color="#FF9800", alpha=0.8)
bars_post = axes[1].bar(x + width/2, post_means, width, label="Post-Training", color="#4CAF50", alpha=0.8)
axes[1].axhline(y=heuristic_mean, color="#888888", linestyle="--", alpha=0.7, label=f"Heuristic ({heuristic_mean:.3f})")
axes[1].set_xticks(x)
axes[1].set_xticklabels([r.capitalize() for r in roles])
axes[1].set_ylabel("Episode Score")
axes[1].set_title("Pre-Training vs Post-Training (per role)")
axes[1].set_ylim(0, 1.0)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")

for bar, val in zip(bars_pre, pre_means):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.2f}",
                 ha="center", va="bottom", fontsize=7)
for bar, val in zip(bars_post, post_means):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.2f}",
                 ha="center", va="bottom", fontsize=7)

# --- Panel 3: Delta improvement ---
deltas = [post - pre for post, pre in zip(post_means, pre_means)]
colors = ["#4CAF50" if d >= 0 else "#F44336" for d in deltas]
bars_d = axes[2].bar(roles, deltas, color=colors, alpha=0.8)
axes[2].axhline(y=0, color="gray", linestyle="-", alpha=0.5)
axes[2].set_ylabel("Score Change (Post - Pre)")
axes[2].set_title("Training Improvement by Role")
axes[2].grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars_d, deltas):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 val + 0.01 if val >= 0 else val - 0.03,
                 f"{val:+.3f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=9)

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to training_results.png")